In [1]:
from pathlib import Path

import pandas as pd

In [2]:
cwd = Path.cwd()
print(cwd)

C:\Users\arya\OneDrive\TA\Coding


In [3]:
mortalitas_df = pd.read_csv(cwd / "data/mortalitas.csv", parse_dates=["Year"], date_format="%Y")
populasi_df = pd.read_csv(cwd / "data/populasi.csv")
bi_rate_df = pd.read_csv(cwd / "data/bi_rate.csv", parse_dates=["Date"], date_format="%d-%m-%y")

In [4]:
mortalitas_df.head()

,Year,Sex,Age,Value
0,1950-01-01,Male,0,0.228515
1,1950-01-01,Female,0,0.185158
2,1950-01-01,Male,1,0.069349
3,1950-01-01,Female,1,0.063058
4,1950-01-01,Male,2,0.044724


In [5]:
mortalitas_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15150 entries, 0 to 15149
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Year    15150 non-null  datetime64[us]
 1   Sex     15150 non-null  str           
 2   Age     15150 non-null  int64         
 3   Value   15150 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 473.6 KB


In [6]:
populasi_df.head()

,Sex,Age,Value
0,Male,45,2013106
1,Male,46,1965078
2,Male,47,1918556
3,Male,48,1880902
4,Male,49,1836713


In [7]:
populasi_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Sex     32 non-null     str  
 1   Age     32 non-null     int64
 2   Value   32 non-null     int64
dtypes: int64(2), str(1)
memory usage: 900.0 bytes


In [8]:
bi_rate_df.head()

,Date,Value
0,2024-12-18,0.0600
1,2024-11-20,0.0600
2,2024-10-16,0.0600
3,2024-09-18,0.0600
4,2024-08-21,0.0625


In [9]:
bi_rate_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    60 non-null     datetime64[us]
 1   Value   60 non-null     float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 1.1 KB


In [10]:
AGE_MIN = min(mortalitas_df["Age"])
AGE_MAX = max(mortalitas_df["Age"])

YEAR_MIN = min(mortalitas_df["Year"])
YEAR_MAX = max(mortalitas_df["Year"])

# Analisis karakteristik data

In [11]:
import seaborn as sns

sns.set_theme(style="whitegrid")

## Plot mortalitas satu usia untuk semua tahun

In [12]:
def plot_usia_vs_tahun(
        age_start=AGE_MIN,
        age_end=AGE_MAX,
        dir=cwd / "plot"
):
    plotname = f"usia vs tahun ({age_start}-{age_end})"
    filepath = dir / f"{plotname}.png"
    if not filepath.exists():
        mask = (mortalitas_df["Age"] >= age_start) & (mortalitas_df["Age"] <= age_end)
        g = sns.FacetGrid(
            mortalitas_df[mask],
            col="Age",
            hue="Sex",
            height=5,
            col_wrap=5,
            sharex=False,

            sharey=False
        )

        g.map_dataframe(sns.lineplot, x="Year", y="Value")

        g.set_titles("Age {col_name}")
        g.set_axis_labels("Year", "Mortality Rate")
        g.add_legend()

        g.tight_layout()
        g.savefig(filepath)
        print(f"Plot {plotname} saved to {filepath}")

In [13]:
plot_usia_vs_tahun(AGE_MIN, 20)
plot_usia_vs_tahun(21, 40)
plot_usia_vs_tahun(41, 60)
plot_usia_vs_tahun(61, 80)
plot_usia_vs_tahun(81, AGE_MAX)

## Plot mortalitas semua usia untuk satu tahun

In [14]:
def plot_tahun_vs_usia(
        year_start=YEAR_MIN,
        year_end=YEAR_MAX,
        dir=cwd / "plot"
):
    plotname = f"tahun vs usia ({year_start}-{year_end})"
    filepath = dir / f"{plotname}.png"
    if not filepath.exists():
        mask = (mortalitas_df["Year"] >= year_start) & (mortalitas_df["Year"] <= year_end)
        df = mortalitas_df.copy()
        df["Year Only"] = df["Year"].dt.year

        g = sns.FacetGrid(
            df[mask],
            col="Year Only",
            hue="Sex",
            height=5,
            col_wrap=5,
            sharex=False,
            sharey=False
        )

        g.map_dataframe(sns.lineplot, x="Age", y="Value")

        g.set_titles("Year {col_name}")
        g.set_axis_labels("Age", "Mortality Rate")
        g.add_legend()

        g.tight_layout()
        g.savefig(filepath)
        print(f"Plot {plotname} saved to {filepath}")

In [15]:
plot_tahun_vs_usia("1950", "1964")
plot_tahun_vs_usia("1965", "1979")
plot_tahun_vs_usia("1980", "1994")
plot_tahun_vs_usia("1995", "2009")
plot_tahun_vs_usia("2010", "2024")

# Pemodelan LocalGLMnet

## Pemodelan

In [16]:
import torch.nn as nn

from ta_module.data import MortalityDataset, MortalityDataModule
from ta_module.models import LocalGLMnet, LocallyConnected2D, MyModel
from ta_module.utils import ElasticNetRegularizationLoss, IdentityTransform
from torch.optim import NAdam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch import seed_everything

import lightning as L

In [17]:
seed_everything(42, workers=True)

Seed set to 42


42

In [18]:
lookback = 10
horizon = 1

mortalitas_dataset_male = MortalityDataset(
    df=mortalitas_df[mortalitas_df["Sex"] == "Male"],
    lookback=lookback,
    horizon=horizon,
    age_col="Age",
    mortality_col="Value",
    year_col="Year"
)

mortalitas_dataset_female = MortalityDataset(
    df=mortalitas_df[mortalitas_df["Sex"] == "Female"],
    lookback=lookback,
    horizon=horizon,
    age_col="Age",
    mortality_col="Value",
    year_col="Year"
)

In [19]:
train_val_test_splits = (0.95, 0.05, 0.0)
batch_size = 16

mortalitas_datamodule = MortalityDataModule(
    mortality_datasets=[mortalitas_dataset_male, mortalitas_dataset_female],
    batch_sizes=batch_size,
    splits=train_val_test_splits
)

In [20]:
ukuran_matriks_input = (lookback, AGE_MAX - AGE_MIN + 1)
lcn_activation_function = nn.Sigmoid()
lcn_kernel_size = 5

create_lcn_layer = LocallyConnected2D.factory(
    input_size=ukuran_matriks_input,
    kernel_size=lcn_kernel_size,
    activation_fn=lcn_activation_function,
    zero_padding=True
)

create_localglmnet_model = LocalGLMnet.factory(
    input_size=ukuran_matriks_input,
    link_fn=IdentityTransform(),
    regression_attention_model=create_lcn_layer()
)

In [21]:
num_ensembles = 10
localglmnet_models = nn.ModuleList([
    create_localglmnet_model()
    for _ in range(num_ensembles)
])

In [22]:
regularization_parameter = 10e-5
train_loss_fns = [
    ElasticNetRegularizationLoss(
        eta=regularization_parameter,
        # Regularisasi LASSO
        alfa=1,
        # Regularisasi diterapkan pada regression attention localglmnet
        model_weights=m.regression_attention_model.parameters()
    )
    for m in localglmnet_models
]

eval_loss_fns = train_loss_fns

In [23]:
optimizers = [
    NAdam(
        params=m.parameters(),
        lr=1e-3
    )
    for m in localglmnet_models
]

lr_schedulers = [
    ReduceLROnPlateau(
        optimizer=o,
        factor=1e-3,
        patience=50,
        min_lr=5e-5,
        cooldown=5,
    )
    for o in optimizers
]

In [24]:
localglmnet_ensembles = [
    MyModel(
        model=m,
        train_loss=tl,
        eval_loss=el,
        optimizer=o,
    )
    for m, tl, el, o in zip(localglmnet_models, train_loss_fns, eval_loss_fns, optimizers)
]

In [25]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [26]:
num_epochs = 500
timezone = ZoneInfo("Asia/Jakarta")
run_datetime = datetime.now(tz=timezone).strftime("%Y-%m-%d_%H-%M-%S")
trainers = [
    L.Trainer(
        max_epochs=num_epochs,
        callbacks=[
            ModelCheckpoint(
                dirpath="best_model_checkpoints",
                filename=f"Model_LocalGLMnet_{i + 1}_{run_datetime}",
                monitor="val_loss",
                mode="min",
                save_top_k=1,
                verbose=False,
            )
        ],
        log_every_n_steps=1,
        deterministic=True
    )
    for i in range(num_ensembles)
]

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For

In [ ]:
# Train ensembles pada data mortalitas laki-laki dan perempuan
print(f"Run datetime: {run_datetime}")
for i, (trainer, model) in enumerate(zip(trainers, localglmnet_ensembles)):
    print("\n================================================================")
    print(f"Training model {i + 1} pada mortalitas laki-laki dan perempuan:")
    print("================================================================\n")
    trainer.fit(
        model=model,
        datamodule=mortalitas_datamodule,
    )

Run datetime: 2026-03-30_12-27-57

Training model 1 pada mortalitas laki-laki dan perempuan:



C:\Users\arya\miniconda3\envs\TA\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\arya\OneDrive\TA\Coding\best_model_checkpoints exists and is not empty.

  | Name       | Type                         | Params | Mode  | FLOPs
----------------------------------------------------------------------------
0 | model      | LocalGLMnet                  | 26.4 K | train | 0    
1 | train_loss | ElasticNetRegularizationLoss | 0      | train | 0    
----------------------------------------------------------------------------
26.4 K    Trainable params
0         Non-trainable params
26.4 K    Total params
0.105     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\arya\miniconda3\envs\TA\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\arya\miniconda3\envs\TA\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
C:\Users\arya\miniconda3\envs\TA\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]